# Static revision — NOT EXECUTED

All historical outputs are cleared. Results in notebook.ipynb do not validate this modified code. Read REVIEW.md for remaining blockers.

- Replaced contiguous ordered splitting with seeded class-wise random splitting.
- Vocabulary is still fitted before splitting, and official IMDb partitions are still combined; these limitations are NOT fixed.
- Padding-aware recurrent state/length handling still requires a coherent redesign.


<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>HW2 @ Deep Learning Course, Dr. Soleymani</b></p>
    <p><b>Desinged by Shaygan Adim</b></p>
</div>

---




*Full Name:* behrad mazaheri

*Student Number:* [student ID removed]

## Homework Overview
In this homework, you will learn to implement, train, and evaluate Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) models on a text classification task using a dataset of IMDB movie reviews, and compare them.

**NOTE : Be sure to answer the analytical questions at the end of the notebook as well.**

In [ ]:
# Imports
import nltk
nltk.download('stopwords')
import os
import random
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk import wordpunct_tokenize
from tqdm import tqdm
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from IPython.display import display, HTML
from datasets import load_dataset

# Enable tqdm progress bar in pandas
tqdm.pandas()

# Set device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dataset

In this section, we’ll load the IMDB dataset and preprocess the data to make it suitable for training RNN and LSTM models.

## Load Dataset
Description of Dataset: The IMDB movie reviews dataset consists of reviews along with their labels (positive or negative sentiment). Each review is a sentence or paragraph of text.

Download the Dataset: We will use a Huggingface to download the dataset into our environment.

In [ ]:
# Load the IMDb dataset (Hugging Face)
dataset = load_dataset("imdb")

# Make sure 'data' folder exists
os.makedirs('data', exist_ok=True)

# Combine train and test splits into one DataFrame
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Save combined DataFrame to CSV
DATA_PATH = 'data/imdb_reviews.csv'
full_df.to_csv(DATA_PATH, index=False)

print(f"✅ Saved combined IMDb reviews to: {DATA_PATH}")
print(f"Total samples: {len(full_df)}")

In [ ]:
# Show 5 random samples from the DataFrame
print(full_df.sample(5))

## Preprocessing

For our models to work effectively, we need to preprocess the text data by cleaning it and converting words to integer indices for training.Preproces steps
such as Tokenization and Cleaning , Replacing Rare Words , Build Vocabulary , Convert Tokens to Indices and Prepare Data for Training.

**NOTE : Do not alter the structure of this preprocessing code, as it aligns with other parts of the notebook.However, minor adjustments for compatibility with your code are allowed if needed.**

In [ ]:
def tokenize(text, stop_words):
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

In [ ]:
def remove_rare_words(tokens, common_tokens, max_len):
    return [token if token in common_tokens
            else '<UNK>' for token in tokens][-max_len:]

In [ ]:
def load_and_preprocess_data(data_path, max_vocab, max_len):
    df = pd.read_csv(data_path)
    stop_words = set(stopwords.words('english'))

    # Clean and tokenize
    df['tokens'] = df['text'].apply(lambda x: tokenize(x, stop_words))

    # Replace rare words with <UNK>
    all_tokens = [token for tokens in df['tokens'] for token in tokens]
    common_tokens = set(list(zip(*Counter(all_tokens).most_common(max_vocab)))[0])
    df['tokens'] = df['tokens'].apply(lambda x: remove_rare_words(x, common_tokens, max_len))

    # Remove sequences with only <UNK>
    df = df[df['tokens'].apply(lambda tokens: any(token != '<UNK>' for token in tokens))]

    # Build vocab
    vocab = sorted(set([token for tokens in df['tokens'] for token in tokens]))
    token2idx = {token: idx for idx, token in enumerate(vocab)}
    token2idx['<PAD>'] = len(token2idx)

    # Index tokens
    df['indexed_tokens'] = df['tokens'].apply(lambda tokens: [token2idx[token] for token in tokens])

    return df['indexed_tokens'].tolist(), df['label'].tolist(), token2idx

In [ ]:
# How many of the most common vocab words to keep
# Uncommon words get replaced with unknown token <UNK>
max_vocab = 2500

# How many tokens long each sequence will be cut to
# Shorter sequences will get the padding token <PAD>
max_len = 100

sequences, targets, token2idx = load_and_preprocess_data(DATA_PATH, max_vocab, max_len)


In [ ]:
def split_data(sequences,targets,valid_ratio=.05,test_ratio=.05,seed=42):
    if valid_ratio<0 or test_ratio<0 or valid_ratio+test_ratio>=1:
        raise ValueError("Split fractions must leave a positive training fraction")
    targets=np.asarray(targets);rng=np.random.default_rng(seed)
    indices={'train':[],'valid':[],'test':[]}
    for label in np.unique(targets):
        group=np.flatnonzero(targets==label);rng.shuffle(group)
        n_test=int(len(group)*test_ratio);n_valid=int(len(group)*valid_ratio)
        indices['test'].extend(group[:n_test]);indices['valid'].extend(group[n_test:n_test+n_valid]);indices['train'].extend(group[n_test+n_valid:])
    output=[]
    for split in ['train','valid','test']:
        group=np.asarray(indices[split],dtype=int);rng.shuffle(group)
        output.extend([[sequences[i] for i in group],targets[group].tolist()])
    return tuple(output)


In [ ]:
train_sequences, train_targets, valid_sequences, valid_targets, test_sequences, test_targets = split_data(sequences, targets)

In [ ]:
def collate(batch):
    inputs, targets = zip(*batch)
    inputs_padded = pad_sequences(inputs, padding_val=token2idx['<PAD>'])
    return torch.LongTensor(inputs_padded), torch.LongTensor(targets)

In [ ]:
def pad_sequences(sequences, padding_val=0, pad_left=False):
    """Pad a list of sequences to the same length with a padding_val."""
    sequence_length = max(len(sequence) for sequence in sequences)
    if not pad_left:
        return [sequence + [padding_val] * (sequence_length - len(sequence)) for sequence in sequences]
    return [[padding_val] * (sequence_length - len(sequence)) + sequence for sequence in sequences]

In [ ]:
batch_size = 256
train_data = list(zip(train_sequences, train_targets))
valid_data = list(zip(valid_sequences, valid_targets))
test_data = list(zip(test_sequences, test_targets))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate)

# RNN section

### RNN with nn.RNN
Implement a basic RNN model using PyTorch's built-in nn.RNN.
Define layers: embedding, RNN, and fully connected.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, n_layers=1,
                 embedding_dimension=50):
        super(RNNClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx=token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.rnn = nn.RNN(embedding_dimension,
                          hidden_size,
                          num_layers=n_layers,
                          batch_first=True,
                          nonlinearity='tanh')
        self.fc = nn.Linear(hidden_size, output_size)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        hidden = torch.zeros(self.n_layers, inputs.size(0), self.hidden_size, device=inputs.device)
        output, hidden = self.rnn(embedded, hidden)
        out = self.fc(hidden[-1])
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.








In [ ]:
rnn_model = RNNClassifier(output_size=2,
                            hidden_size=128,
                            vocab_size=len(token2idx),
                            device=device,
                            n_layers=2,
                            embedding_dimension=100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)

In [ ]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, valid_loader, optimizer, criterion, num_epochs=6):
    history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]")
        for inputs, labels in pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100 * correct / total:.2f}%")
        train_loss = total_loss / total
        train_acc = correct / total
        valid_loss, valid_acc = evaluate_model(model, valid_loader, criterion)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)
        print(f"Epoch {epoch}/{num_epochs} - train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, valid_loss: {valid_loss:.4f}, valid_acc: {valid_acc:.4f}")
    return history

rnn_history = train_model(rnn_model, train_loader, valid_loader, optimizer, criterion, num_epochs=6)


### RNN from Scratch
Implement an RNN from scratch by creating a custom RNN cell and a model that stacks these cells over time.

In [ ]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomRNNCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        combined = torch.cat([input, hidden], dim=1)
        hidden = torch.tanh(self.i2h(combined))
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return hidden


In [ ]:
class CustomRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.rnn_cell = CustomRNNCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        hidden = torch.zeros(inputs.size(0), self.hidden_size, device=inputs.device)
        for t in range(embedded.size(1)):
            hidden = self.rnn_cell(embedded[:, t, :], hidden)
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.


### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [ ]:
custom_rnn_model = CustomRNN(vocab_size=len(token2idx),
                              embedding_dim=100,
                              hidden_size=128,
                              output_size=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(custom_rnn_model.parameters(), lr=0.001)


In [ ]:
custom_rnn_history = train_model(custom_rnn_model, train_loader, valid_loader, optimizer, criterion, num_epochs=6)


### evaluate RNN models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation. This function can be used to evaluate the performance LSTM models too.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each rnn models achieves at least 70% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [ ]:
# Evaluate on test data
def evaluate_on_test(model, test_loader):
    model.eval()
    y_true_test = []
    y_pred_test = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            y_true_test.extend(labels.cpu().tolist())
            y_pred_test.extend(preds.cpu().tolist())
    accuracy = sum(int(y == p) for y, p in zip(y_true_test, y_pred_test)) / len(y_true_test)
    print(f"Test Accuracy: {accuracy * 100:.2f}%")
    print(classification_report(y_true_test, y_pred_test))


In [ ]:
print("Evaluating RNN models on the test dataset")
print("nn.RNN model results:")
evaluate_on_test(rnn_model, test_loader)
print("Custom RNN model results:")
evaluate_on_test(custom_rnn_model, test_loader)


# LSTM section

### LSTM with nn.LSTM
Define an LSTM model using PyTorch's built-in nn.LSTM.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, bidirectional=False, n_layers=1,
                 embedding_dimension=50):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device
        self.bidirectional = bidirectional

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx = token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.lstm = nn.LSTM(embedding_dimension,
                            hidden_size,
                            num_layers=n_layers,
                            batch_first=True,
                            bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), output_size)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        num_directions = 2 if self.bidirectional else 1
        hidden = torch.zeros(self.n_layers * num_directions, inputs.size(0), self.hidden_size, device=inputs.device)
        cell_state = torch.zeros(self.n_layers * num_directions, inputs.size(0), self.hidden_size, device=inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        output, (hidden, cell_state) = self.lstm(embedded, (hidden, cell_state))
        if self.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return out  # probabilities for each class in the output.


### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.

In [ ]:
lstm_model = LSTMClassifier(output_size=2,
                             hidden_size=128,
                             vocab_size=len(token2idx),
                             device=device,
                             bidirectional=False,
                             n_layers=2,
                             embedding_dimension=100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)


In [ ]:
lstm_history = train_model(lstm_model, train_loader, valid_loader, optimizer, criterion, num_epochs=6)


### Custom LSTM from Scratch
Implement an LSTM from scratch by defining a LSTM cell and a model that combines these cells over the sequence.

In [ ]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomLSTMCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.linear = nn.Linear(input_size + hidden_size, hidden_size * 4)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden, cell_state):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        combined = torch.cat([input, hidden], dim=1)
        gates = self.linear(combined)
        input_gate, forget_gate, candidate, output_gate = gates.chunk(4, dim=1)
        input_gate = torch.sigmoid(input_gate)
        forget_gate = torch.sigmoid(forget_gate)
        candidate = torch.tanh(candidate)
        output_gate = torch.sigmoid(output_gate)
        cell_state = forget_gate * cell_state + input_gate * candidate
        hidden = output_gate * torch.tanh(cell_state)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return hidden, cell_state # New hidden state , New cell state


In [ ]:
class CustomLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.lstm_cell = CustomLSTMCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        hidden = torch.zeros(inputs.size(0), self.hidden_size, device=inputs.device)
        cell_state = torch.zeros(inputs.size(0), self.hidden_size, device=inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        for t in range(embedded.size(1)):
            hidden, cell_state = self.lstm_cell(embedded[:, t, :], hidden, cell_state)
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return out  # probabilities for each class in the output.


### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [ ]:
custom_lstm_model = CustomLSTM(vocab_size=len(token2idx),
                                embedding_dim=100,
                                hidden_size=128,
                                output_size=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(custom_lstm_model.parameters(), lr=0.001)


In [ ]:
custom_lstm_history = train_model(custom_lstm_model, train_loader, valid_loader, optimizer, criterion, num_epochs=6)


### evaluate LSTM models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation.
you can use the `evaluate_on_test` function implemented in the previous section. Alternatively, you may write a new function to conduct this evaluation.ensure report the "classification_report" of both LSTM models.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each lstm models achieves at least 80% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [ ]:
print("Evaluating LSTM models on the test dataset")
print("nn.LSTM model results:")
evaluate_on_test(lstm_model, test_loader)
print("Custom LSTM model results:")
evaluate_on_test(custom_lstm_model, test_loader)


# Testing RNN and LSTM Models on a New Review

In [ ]:
# Example review
review = "It is no wonder that the film has such a high rating, it is quite literally breathtaking. What can I say that hasn't said before? Not much, it's the story, the acting, the premise, but most of all, this movie is about how it makes you feel. Sometimes you watch a film, and can't remember it days later, this film loves with you, once you've seen it, you don't forget."


## Preprocess the test Review
To prepare the review for the model, we need to follow similar preprocessing steps as we did for the dataset:

Remove special characters and convert the text to lowercase.
Tokenize the text into individual words.
Remove stopwords to focus only on meaningful words.
Convert tokens to indices based on the token2idx dictionary created earlier.
Pad or truncate the sequence to a length of max_len .


In [ ]:
import torch
import re
from nltk import wordpunct_tokenize
from nltk.corpus import stopwords


# Preprocessing function
def preprocess_text(text, stop_words, token2idx, max_len):

    ########################### TODO ###########################
    # Step 1: Clean and lowercase the input text
    ################################################################

    text = re.sub(r'[^\w\s]', '', text).lower()

    ########################### TODO ###########################
    # Step 2: Tokenize the text into words
    # - Use wordpunct_tokenize to split the cleaned text into individual word tokens.
    ############################################################

    tokens = wordpunct_tokenize(text)

    ########################### TODO ###########################
    # Step 3: Remove stopwords from the token list
    ############################################################

    tokens = [token for token in tokens if token not in stop_words]

    ########################### TODO ###########################
    # Step 4: Convert tokens to indices based on the token2idx dictionary
    # - For each token in the list, get the corresponding index from the token2idx dictionary.
    # - If a token is not found in token2idx, replace it with the index of '<UNK>'.
    ################################################################

    unk_index = token2idx.get('<UNK>', token2idx['<PAD>'])
    tokens_idx = [token2idx.get(token, unk_index) for token in tokens]

    ########################### TODO ###########################
    # Step 5: Pad or truncate the tokens_idx list to the desired max_len
    ############################################################

    if len(tokens_idx) < max_len:
        tokens_idx = tokens_idx + [token2idx['<PAD>']] * (max_len - len(tokens_idx))
    else:
        tokens_idx = tokens_idx[:max_len]

    ########################### End of TODOs ###########################

    return tokens_idx  # Return the processed list of indices

# Get stopwords
stop_words = set(stopwords.words('english'))

########################### TODO ###########################
# Preprocess the review
review_indices = preprocess_text(review, stop_words, token2idx, max_len)
############################################################

# Convert the indices to a tensor and move it to the device (GPU or CPU)
input_tensor = torch.LongTensor([review_indices]).to(device)


## Make Predictions
Now that we have preprocessed the review, use both the RNN and LSTM models to make predictions on the sentiment of the review.

Set the model to evaluation mode to prevent updates during inference.
Predict the sentiment class by passing the input_tensor to the model.
Interpret the prediction as either "Positive" or "Negative" based on the model's output.

In [ ]:
def predict_sentiment(model, input_tensor, model_name="Model"):
    model.eval()  # Set the model to evaluation mode
    ############################# TODO #############################
    # TODO: Perform a forward pass with the model on the input_tensor,
    # get the predicted class label, and map it to "Positive" or "Negative".
    ################################################################
    with torch.no_grad():
        outputs = model(input_tensor)
        predicted_class = torch.argmax(outputs, dim=1).item()
        class_label = "Positive" if predicted_class == 1 else "Negative"

    print(f"The predicted class for the review by {model_name} is: {class_label}")


In [ ]:
print("Prediction with nn.RNN model:")
predict_sentiment(rnn_model, input_tensor, model_name="nn.RNN")

print("Prediction with CustomRNN model:")
predict_sentiment(custom_rnn_model, input_tensor, model_name="CustomRNN")

print("Prediction with nn.LSTM model:")
predict_sentiment(lstm_model, input_tensor, model_name="nn.LSTM")

print("Prediction with CustomLSTM model:")
predict_sentiment(custom_lstm_model, input_tensor, model_name="CustomLSTM")


# Questions

[1] - Based on your observations, what do you think caused the difference in performance between the RNN and LSTM models (on test set)? Analyze this difference using the results from the notebook, and discuss where a simple RNN might perform better.

[2] - If we increase max_len in the preprocessing step to 300, what changes in models (rnn & lstm ) performance would you expect, and why? Please *explain* and discuss the impact this may have on the learning process and the final results.

1. 
The LSTM usually performs better because it is designed to handle long-term dependencies through its gating mechanisms. The input, forget, and output gates allow the model to control what information should be stored, updated, or forgotten. A simple RNN repeatedly applies the same recurrent transformation, so its gradients can vanish or explode over long sequences. This makes it harder for the simple RNN to preserve useful information from earlier words in the review. A simple RNN might perform better or be sufficient when the sequences are short, the task depends mostly on local patterns, or the dataset is small enough that a simpler model generalizes better.

2. 
Increasing max_len to 300 allows the models to see more of each review, so potentially useful information from later parts of the text will not be truncated. This could improve performance if important sentiment cues appear near the end of reviews. However, it also makes training harder and slower because the recurrent model must process more time steps. For a simple RNN, performance may decrease or improve only slightly because longer sequences make vanishing gradients more severe. The LSTM is more likely to benefit from the longer input because its gates help preserve relevant information over many time steps. Still, if many added tokens are uninformative, both models may become slower and possibly more prone to overfitting or noise.